In [4]:
import os
import json
from glob import glob
import pandas as pd

'''
The script loads each JSON, extracts the time base (Simulation.sim_time) and (if present) anomaly_time, then builds a flat table with sensors S1–S8, 
pneumatic valves (AV1–AV3, AV2_PP/LD, AV3_PP/LD), and PID controllers. For PID1–PID3, it uses the Header and Values arrays to create columns (I, D, 
P, E, IN, OUT, ONoff). Anomaly flags (VM1–VM7) are populated by mapping Simulation.anomaly and its anomaly_time window onto the aligned timeline. 
Finally, the script writes one .xlsx per JSON (same basename), sheet “Sheet1”, preserving the index (“Unnamed: 0”).


BASE_DIR is the root folder the script uses: it reads all input JSON files from this directory and saves the generated .xlsx outputs back into it 
(here set to Working_conditions/working_condition1 as example; change it to your working directory if needed).
'''
BASE_DIR = "/kaggle/input/oil-and-gas/Data/Working_conditions"

OUT_DIR = "/kaggle/working/excel_output"
os.makedirs(OUT_DIR, exist_ok=True)

def json_to_target_xlsx(json_path: str) -> str:
    with open(json_path, "r") as f:
        js = json.load(f)
    
    sim = js.get("Simulation", {})
    sensors = js.get("Sensors", {})
    valves = js.get("Pneumatic_Valves", {})
    pid = js.get("PID_Controllers", {})
    
    # Time
    time = sim.get("sim_time", [])
    n = len(time)
    df = pd.DataFrame({"Time": time})
    
    # Sensors S1..S8
    for s in [f"S{i}" for i in range(1,9)]:
        if s in sensors:
            df[s] = sensors[s]
    
    # Valves (AV1, AV2, AV3, AV2_PP, AV2_LD, AV3_PP, AV3_LD) if present
    for k in ["AV1","AV2","AV3","AV2_PP","AV2_LD","AV3_PP","AV3_LD"]:
        v = valves.get(k)
        if isinstance(v, list) and len(v) == n:
            df[k] = v
    
    # PID controllers: PID1..PID3 with Header+Values
    for i in [1,2,3]:
        pid_key = f"PID{i}"
        if pid_key in pid:
            header = pid[pid_key].get("Header", [])
            values = pid[pid_key].get("Values", [])
            # Build DataFrame with same length as n
            if values and len(values) == n:
                pid_df = pd.DataFrame(values, columns=header)
                # Ensure column order P/I/D/E/IN/OUT/ONoff? Keep header as is
                for col in header:
                    df[col] = pid_df[col].values
    
    # VM flags: VM1..VM7
    # Initialize zeros
    for vm in [f"VM{i}" for i in range(1,8)]:
        df[vm] = 0
    # If anomaly tag exists, map its anomaly_time to the corresponding VM column
    anomaly_name = sim.get("anomaly")
    anomaly_time = sim.get("anomaly_time")
    if isinstance(anomaly_name, str) and isinstance(anomaly_time, list) and len(anomaly_time) > 0:
        # Clip/pad to n
        arr = (anomaly_time[:n] + [0]*n)[:n]
        vm_col = anomaly_name if anomaly_name in df.columns else None
        if vm_col is None and anomaly_name.startswith("VM"):
            vm_col = anomaly_name  # create if not exists
            if vm_col not in df.columns:
                df[vm_col] = 0
        if vm_col:
            df[vm_col] = arr
    
    # Save with same basename
    base = os.path.splitext(os.path.basename(json_path))[0]
    # out_path = os.path.join(BASE_DIR, f"{base}.xlsx")
    out_path = os.path.join(OUT_DIR, f"{base}.xlsx")
    # Write as single sheet; include index to mimic 'Unnamed: 0'
    df.to_excel(out_path, index=True, sheet_name="Sheet1")
    return out_path

# # Demo on the provided files
# outputs = [json_to_target_xlsx(p) for p in sorted(glob(os.path.join(BASE_DIR, "*.json")))]
outputs = []

for wc_dir in sorted(glob(os.path.join(BASE_DIR, "working_condition*"))):
    wc_name = os.path.basename(wc_dir)

    out_wc_dir = os.path.join(OUT_DIR, wc_name)
    os.makedirs(out_wc_dir, exist_ok=True)

    for json_file in sorted(glob(os.path.join(wc_dir, "*.json"))):
        out_path = json_to_target_xlsx(json_file)
        outputs.append(out_path)

outputs

['/kaggle/working/excel_output/working_condition1.xlsx',
 '/kaggle/working/excel_output/working_condition10.xlsx',
 '/kaggle/working/excel_output/working_condition11.xlsx',
 '/kaggle/working/excel_output/working_condition12.xlsx',
 '/kaggle/working/excel_output/working_condition13.xlsx',
 '/kaggle/working/excel_output/working_condition14.xlsx',
 '/kaggle/working/excel_output/working_condition15.xlsx',
 '/kaggle/working/excel_output/working_condition16.xlsx',
 '/kaggle/working/excel_output/working_condition17.xlsx',
 '/kaggle/working/excel_output/working_condition18.xlsx',
 '/kaggle/working/excel_output/working_condition19.xlsx',
 '/kaggle/working/excel_output/working_condition2.xlsx',
 '/kaggle/working/excel_output/working_condition20.xlsx',
 '/kaggle/working/excel_output/working_condition21.xlsx',
 '/kaggle/working/excel_output/working_condition22.xlsx',
 '/kaggle/working/excel_output/working_condition23.xlsx',
 '/kaggle/working/excel_output/working_condition24.xlsx',
 '/kaggle/workin

In [2]:
import shutil
import os

OUT_DIR = "/kaggle/working/excel_output"

if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
    print("Output folder deleted.")
else:
    print("Output folder does not exist.")


Output folder deleted.


In [5]:
!zip -r /kaggle/working/excel_output.zip /kaggle/working/excel_output

  adding: kaggle/working/excel_output/ (stored 0%)
  adding: kaggle/working/excel_output/working_condition6/ (stored 0%)
  adding: kaggle/working/excel_output/working_condition23/ (stored 0%)
  adding: kaggle/working/excel_output/working_condition10/ (stored 0%)
  adding: kaggle/working/excel_output/working_condition44.xlsx (deflated 7%)
  adding: kaggle/working/excel_output/working_condition42.xlsx (deflated 8%)
  adding: kaggle/working/excel_output/working_condition44/ (stored 0%)
  adding: kaggle/working/excel_output/working_condition2.xlsx (deflated 5%)
  adding: kaggle/working/excel_output/working_condition19/ (stored 0%)
  adding: kaggle/working/excel_output/working_condition18/ (stored 0%)
  adding: kaggle/working/excel_output/working_condition38.xlsx (deflated 7%)
  adding: kaggle/working/excel_output/working_condition38/ (stored 0%)
  adding: kaggle/working/excel_output/working_condition5.xlsx (deflated 4%)
  adding: kaggle/working/excel_output/working_condition45/ (stored 0%)

In [12]:
!ls -lh /kaggle/working


total 19M
drwxr-xr-x 48 root root 4.0K Feb  3 03:03 excel_output
-rw-r--r--  1 root root  19M Feb  3 03:06 excel_output.zip


In [ ]:
import os
import json
from glob import glob
import pandas as pd

# =========================
# PATH
# =========================
BASE_DIR = "/kaggle/input/oil-and-gas/Data/Test_with_anomalies"
OUT_DIR  = "/kaggle/working/excel_anomaly_output"

os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# MAIN FUNCTION
# =========================
def json_to_anomaly_xlsx(json_path):

    with open(json_path, "r") as f:
        js = json.load(f)

    sim     = js.get("Simulation", {})
    sensors = js.get("Sensors", {})
    valves  = js.get("Pneumatic_Valves", {})
    pid     = js.get("PID_Controllers", {})

    # =========================
    # TIME
    # =========================
    time = sim.get("sim_time", [])
    n = len(time)

    df = pd.DataFrame({"Time": time})

    # =========================
    # SENSORS S1–S8
    # =========================
    for s in [f"S{i}" for i in range(1,9)]:
        if s in sensors:
            df[s] = sensors[s]

    # =========================
    # VALVES
    # =========================
    valve_keys = ["AV1","AV2","AV3","AV2_PP","AV2_LD","AV3_PP","AV3_LD"]

    for k in valve_keys:
        v = valves.get(k)
        if isinstance(v, list) and len(v) == n:
            df[k] = v

    # =========================
    # PID CONTROLLERS
    # =========================
    for i in [1,2,3]:
        pid_key = f"PID{i}"

        if pid_key in pid:
            header = pid[pid_key].get("Header", [])
            values = pid[pid_key].get("Values", [])

            if values and len(values) == n:
                pid_df = pd.DataFrame(values, columns=header)

                for col in header:
                    df[col] = pid_df[col].values

    # =========================
    # ANOMALY FLAGS (VM1–VM7)
    # =========================
    for vm in [f"VM{i}" for i in range(1,8)]:
        df[vm] = 0

    anomaly_name = sim.get("anomaly")
    anomaly_time = sim.get("anomaly_time")

    if isinstance(anomaly_name, str) and isinstance(anomaly_time, list):

        if anomaly_name.startswith("VM"):

            vm_col = anomaly_name

            # Samakan panjang dengan time
            anomaly_time = anomaly_time[:n]

            if len(anomaly_time) < n:
                anomaly_time += [0]*(n-len(anomaly_time))

            df[vm_col] = anomaly_time

    # =========================
    # LABEL TAMBAHAN (PENTING)
    # =========================
    df["label"] = 1  # anomaly

    # ambil nama folder (contoh: anomalyVM1_medium)
    folder_name = os.path.basename(os.path.dirname(json_path))

    df["anomaly_type"] = folder_name

    # =========================
    # SAVE (MIRROR FOLDER STRUCTURE)
    # =========================
    relative_path = os.path.relpath(json_path, BASE_DIR)
    out_path = os.path.join(OUT_DIR, relative_path.replace(".json", ".xlsx"))

    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    df.to_excel(out_path, index=True, sheet_name="Sheet1")

    return out_path


# =========================
# PROCESS ALL (RECURSIVE)
# =========================
outputs = []

json_files = glob(os.path.join(BASE_DIR, "**", "*.json"), recursive=True)

for json_file in sorted(json_files):
    out = json_to_anomaly_xlsx(json_file)
    outputs.append(out)

print("Total files processed:", len(outputs))